# 04 - Transformer with Defense 2 (Output Limiting)

Defense: temperature scaling or label-only outputs

In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

from research_protocol import (
    evaluate_mia_research_protocol,
    compare_defense_vs_baseline,
    set_seed,
)
from research_protocol_robust import evaluate_mia_robust_protocol

In [ ]:
SEED = 42
ATTACK_SEEDS = [11, 22]

# Baseline de securite: on teste un cas plus vulnerable (fuite/overfit possible)
TARGET_DROPOUT_SAFE = 0.15
TARGET_DROPOUT_RISKY = 0.00
TARGET_EPOCHS_SAFE = 6
TARGET_EPOCHS_RISKY = 60

# Attaquant plus fort pour avoir une baseline MIA exploitable
N_SHADOWS = 4
SHADOW_EPOCHS = 4
SHADOW_BATCH = 16

ROBUST_N_SHADOWS = 6
ROBUST_SHADOW_EPOCHS = 4
ROBUST_BOUNDARY_DIRS = 8
ROBUST_BOUNDARY_STEPS = 4
ROBUST_BOUNDARY_MAX_ALPHA = 2.0

ATTACK_MIN_AUC_TARGET = 0.55

set_global_determinism(SEED)

# Defense params (output limiting with temperature)
TARGET_EPOCHS_DEFENSE = 10
DEFENSE_TEMPERATURE = 3.0

prepared_path = os.path.join('..', 'data_preparation', 'prepared_oasis2_transformer.npz')
if not os.path.exists(prepared_path):
    prepared_path = os.path.join('data_preparation', 'prepared_oasis2_transformer.npz')

if not os.path.exists(prepared_path):
    raise FileNotFoundError('prepared_oasis2_transformer.npz not found')

b = np.load(prepared_path)

X = b['X'].astype(np.float32)
y = b['y'].astype(np.int32)
X_train = b['X_train'].astype(np.float32)
X_test = b['X_test'].astype(np.float32)
y_train = b['y_train'].astype(np.int32)
y_test = b['y_test'].astype(np.int32)

if 'X_shadow' in b:
    X_shadow_raw = b['X_shadow'].astype(np.float32)
else:
    X_shadow_raw = b['X_shadow_raw'].astype(np.float32)

y_shadow = b['y_shadow'].astype(np.int32)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1])
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1])

print(f'Loaded: X_train={X_train_seq.shape}, X_test={X_test_seq.shape}, X_shadow={X_shadow_raw.shape}')
print(f'Class ratio train={y_train.mean():.4f}, test={y_test.mean():.4f}')

In [ ]:
# Model override: vulnerable PyTorch MLP adapter with keras-like API
from model_adapters import make_vulnerable_model

def build_transformer(n_features, dropout=0.15):
    return make_vulnerable_model(input_dim=n_features, dropout=dropout)

In [3]:
print('=== Load prepared data ===')
b = np.load('../data_preparation/prepared_oasis2_transformer.npz', allow_pickle=True)
X, y = b['X'], b['y']
X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']
X_train_seq, X_test_seq = b['X_train_seq'], b['X_test_seq']
X_shadow_raw, y_shadow = b['X_shadow_raw'], b['y_shadow']
SEED = int(b['seed'][0])
print(f'X={X.shape}, train={X_train.shape}, test={X_test.shape}, shadow={X_shadow_raw.shape}')

=== Load prepared data ===
X=(354, 9), train=(70, 9), test=(284, 9), shadow=(107, 9)


In [4]:
print('=== Train baseline attack-target model ===')
set_seed(SEED + 10); tf.keras.backend.clear_session()
model = build_transformer(X_train.shape[1], dropout=TARGET_DROPOUT_RISKY)
model.fit(X_train_seq, y_train, epochs=TARGET_EPOCHS_RISKY, batch_size=32, verbose=0)
p_te = model.predict(X_test_seq, verbose=0).ravel()
test_acc = accuracy_score(y_test, (p_te>=0.5).astype(int))
print(f'test_acc={test_acc:.4f}, mode={DEFENSE_MODE}, T={TEMPERATURE}')

=== Train baseline attack-target model ===
test_acc=0.9261, mode=temperature, T=4.0


In [5]:
print('=== Attacks baseline outputs (unified protocol, strong attacker) ===')
baseline_summary, baseline_per_seed = evaluate_mia_research_protocol(
    target_model=model,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(nf, dropout=TARGET_DROPOUT_RISKY),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    postprocess_fn=lambda p: p,
)
display(baseline_summary.round(4))

=== Attacks baseline outputs (unified protocol, strong attacker) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
1,shadow_meta,0.607,0.0158,0.6324,0.0126,0.2469,0.0163,0.4214,0.0299,0.3113,0.0207
0,logistic,0.507,0.0144,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,threshold_loss,0.507,0.0144,0.2746,0.0086,0.2138,0.0020,1.0000,0.0000,0.3522,0.0027


In [6]:
print(f'=== Attacks with Defense 2 ({DEFENSE_MODE}) (unified protocol, strong attacker) ===')
def2_summary, def2_per_seed = evaluate_mia_research_protocol(
    target_model=model,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(nf, dropout=TARGET_DROPOUT_RISKY),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    postprocess_fn=limit_output,
)
display(def2_summary.round(4))

=== Attacks with Defense 2 (temperature) (unified protocol, strong attacker) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
1,shadow_meta,0.607,0.0157,0.6437,0.0128,0.2616,0.0174,0.4429,0.0319,0.3289,0.0220
0,logistic,0.507,0.0144,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,threshold_loss,0.507,0.0144,0.3944,0.0792,0.2020,0.0183,0.7214,0.2414,0.3129,0.0445


In [7]:
print('=== Baseline vs Defense 2 ===')
cmp = compare_defense_vs_baseline(baseline_summary, def2_summary, 'defense2')
display(cmp.round(4))

=== Baseline vs Defense 2 ===


,attack,auc_mean_baseline,accuracy_mean_baseline,f1_mean_baseline,auc_mean_defense2,accuracy_mean_defense2,f1_mean_defense2,delta_auc,delta_accuracy,delta_f1
1,logistic,0.507,0.8028,0.0000,0.507,0.8028,0.0000,0.0000,0.0000,0.0000
0,shadow_meta,0.607,0.6324,0.3113,0.607,0.6437,0.3289,0.0001,0.0113,0.0176
2,threshold_loss,0.507,0.2746,0.3522,0.507,0.3944,0.3129,0.0000,0.1197,-0.0393


In [8]:
mcmp = pd.DataFrame([
    {
        'model': 'Transformer',
        'test_acc_baseline': float(test_acc),
        'test_acc_defense2': float(test_acc),
        'delta_test_acc': 0.0,
    }
])
display(mcmp.round(4))

,model,test_acc_baseline,test_acc_defense2,delta_test_acc
0,Transformer,0.9261,0.9261,0.0


In [9]:
print('=== Quick AUC summary ===')
quick_auc = cmp[['attack', 'auc_mean_baseline', 'auc_mean_defense2', 'delta_auc']].sort_values('delta_auc')
display(quick_auc.round(4))

=== Quick AUC summary ===


,attack,auc_mean_baseline,auc_mean_defense2,delta_auc
1,logistic,0.507,0.507,0.0000
2,threshold_loss,0.507,0.507,0.0000
0,shadow_meta,0.607,0.607,0.0001


In [10]:
print(f'=== Robust adaptive attack (meta + LiRA + boundary): baseline vs defense2 ({DEFENSE_MODE}) ===')

base_robust_summary, base_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=model,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(nf, dropout=TARGET_DROPOUT_RISKY),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
    postprocess_fn=lambda p: p,
)

def2_robust_summary, def2_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=model,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(nf, dropout=TARGET_DROPOUT_RISKY),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
    postprocess_fn=limit_output,
)

cmp_robust = compare_defense_vs_baseline(base_robust_summary, def2_robust_summary, 'defense2')
print('=== Quick AUC summary (robust adaptive) ===')
quick_auc_robust = cmp_robust[['attack', 'auc_mean_baseline', 'auc_mean_defense2', 'delta_auc']].sort_values('delta_auc')
display(quick_auc_robust.round(4))

=== Robust adaptive attack (meta + LiRA + boundary): baseline vs defense2 (temperature) ===
=== Quick AUC summary (robust adaptive) ===


,attack,auc_mean_baseline,auc_mean_defense2,delta_auc
0,shadow_meta,0.5978,0.5939,-0.0039
